In [ ]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

In [ ]:
#Zero-shot classifier for Q1, Q2, Q6
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

In [ ]:
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

# Probability assigned to the correct answer
ground_truth_score = result["scores"][result["labels"].index(ans_150)]

print(round(ground_truth_score, 3))

In [ ]:
# Embed the prompt
query_embedding = model.encode(
    [prompt_150],
    convert_to_numpy=True
).astype("float32")

# Search top 10
distances, indices = index.search(query_embedding, k=10)

print("Retrieved indices:", indices[0])

# Find the rank of the true document (index 150)
if 150 in indices[0]:
    rank = list(indices[0]).index(150) + 1
    print("Rank of true document:", rank)
else:
    print("True document not found in top 10")

In [ ]:
# Retrieve top-10 from FAISS
query_embedding = model.encode(
    [prompt_150],
    convert_to_numpy=True
).astype("float32")

distances, indices = index.search(query_embedding, k=10)
retrieved_indices = indices[0]

# Load Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Prepare document pairs
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]

# Predict relevance scores
ce_scores = cross_encoder.predict(pairs)

# Sort by Cross-Encoder score (highest first)
reranked = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Reranked Results:")
for rank, (idx, score) in enumerate(reranked, start=1):
    print(f"{rank:2d}. Index={idx:4d}  Score={score:.4f}")

# Find the new rank of the true document (index 150)
for rank, (idx, score) in enumerate(reranked, start=1):
    if idx == 150:
        print(f"\nTrue document reranked to position: {rank}")
        print(f"Cross-Encoder score: {score:.4f}")
        break

In [ ]:
from transformers import AutoTokenizer

# Row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Retrieve top-5 documents
query_embedding = model.encode(
    [prompt_42],
    convert_to_numpy=True
).astype("float32")

_, indices = index.search(query_embedding, k=5)

docs = [kb[i] for i in indices[0]]
concatenated_docs = " ".join(docs)

# Build input string
text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Tokenize without truncation
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer(
    text,
    truncation=False,
    add_special_tokens=True
)

print(len(tokens["input_ids"]))

In [ ]:
# True document from the KB
true_document = kb[150]

# Create RAG prompt
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

# Run zero-shot classification
result_rag = zs(
    rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

# Score assigned to the ground-truth answer
score_dict = dict(zip(result_rag["labels"], result_rag["scores"]))
ground_truth_score = score_dict[ans_150]

print(round(ground_truth_score, 3))

In [ ]:
# Use an unrelated document as the context
wrong_document = kb[999]

# Create the adversarial RAG prompt
adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

# Run the zero-shot classifier
result_adv = zs(
    adversarial_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

# Probability assigned to the ground-truth answer
score_dict = dict(zip(result_adv["labels"], result_adv["scores"]))
ground_truth_score = score_dict[ans_150]

print(round(ground_truth_score, 3))

In [ ]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])

    # Correct answer text
    correct_doc = str(row[row["answer"]])

    # Retrieve top-5 documents
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True
    ).astype("float32")

    _, indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    # Check if the exact correct string appears in any retrieved document
    if any(correct_doc in doc for doc in retrieved_docs):
        hits += 1

hit_rate = hits / 100 * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1))

In [ ]:
from sentence_transformers import CrossEncoder

# Load Cross-Encoder once
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def map_at_3(predicted, actual):
    """
    predicted: list of option letters sorted by confidence (e.g. ['C','A','E'])
    actual: correct option letter (e.g. 'A')
    """
    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)
    return 0.0

scores = []

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    correct_letter = row["answer"]

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    # -------------------
    # Retrieve
    # -------------------
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True
    ).astype("float32")

    _, indices = index.search(query_embedding, 5)

    retrieved_indices = indices[0]
    retrieved_docs = [kb[idx] for idx in retrieved_indices]

    # -------------------
    # Rerank
    # -------------------
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # -------------------
    # Augment
    # -------------------
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # -------------------
    # Predict
    # -------------------
    result = zs(
        rag_prompt,
        candidate_labels=labels,
        multi_label=False
    )

    # Convert predicted texts back to option letters
    text_to_letter = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }

    ranked_letters = [
        text_to_letter[label]
        for label in result["labels"]
    ]

    top3 = ranked_letters[:3]

    scores.append(map_at_3(top3, correct_letter))

print("Average MAP@3 =", round(np.mean(scores), 3))